# C2 · Common ArcFace warmup

Mỗi seed có một shared IR-18/ArcFace warmup. Common checkpoint được chọn bằng nominal dev TAR ở target FMR $10^{-2}$. `tau_train.json` ghi threshold fit trên tập con dev khác và SHA-256 của checkpoint. Run resume tự động khi có `last.pt`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
REPO = DRIVE / "C2_Base"
WORK = DRIVE / "C2_work"
DATA_ROOT = Path("/content/digiface72")  # Extracted, authorized 72-capture partition
MANIFEST = WORK / "manifests/digiface72.csv"

assert (REPO / "configs/pilot.yaml").is_file(), f"Set REPO to the copied C2_Base source folder: {REPO}"
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")

def run(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], cwd=REPO, check=True)

In [ ]:
import torch

DEVICE = "cuda:0"
assert torch.cuda.is_available(), "Select a Colab GPU runtime before training."
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30, "GiB")

In [ ]:
CONFIG = REPO / "configs/pilot.yaml"
SCOPE = "pilot"  # Use "full" only for the preregistered 6K-ID follow-up.
SEEDS = [17]  # Use the same list in notebooks 03 and 04.
AMP_MODE = "off"  # Select "fp16" or "bf16" to run its CUDA numerical gate.
assert MANIFEST.is_file() and DATA_ROOT.is_dir()

if AMP_MODE == "bf16":
    assert torch.cuda.is_bf16_supported(), "This GPU does not support bfloat16."
if AMP_MODE != "off":
    run("pytest", "-q", "tests/test_models.py", "-m", "cuda", "-k", AMP_MODE)

In [ ]:
for seed in SEEDS:
    run_dir = WORK / "runs" / SCOPE / f"seed-{seed}" / "warmup"
    resume = ["--resume"] if (run_dir / "last.pt").exists() else []
    run("scripts.train", "warmup", "--config", CONFIG,
        "--manifest", MANIFEST, "--data-root", DATA_ROOT, "--out", run_dir,
        "--scope", SCOPE, "--seed", seed, "--amp", AMP_MODE, "--device", DEVICE, *resume)
    print((run_dir / "tau_train.json").read_text(encoding="utf-8"))